In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os
import warnings

warnings.filterwarnings('ignore')

# ۱. تنظیمات مسیرها و متغیرها
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output1.xlsx'

os.makedirs(os.path.dirname(output_filename), exist_ok=True)

all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
                'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# ۲. خواندن فایل اصلی
try:
    df_raw = pd.read_excel(file_path)
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.sort_values(by='date')
    print("✅ فایل ورودی با موفقیت خوانده شد.")
except Exception as e:
    print(f"❌ خطا در خواندن فایل: {e}")
    exit()

# ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[all_features])

dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(scaled_data)

cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

def calculate_distance(i):
    label, point = labels[i], scaled_data[i].reshape(1, -1)
    if label != -1:
        return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
    return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

# اصلاح خطای سینتکسی (افزودن علامت ضرب )
df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# ۴. تعریف بازه‌ها (بازه نرمال و بازه تست ۳۰ روز آخر)
last_date = df_cleaned.index.max()
start_analysis_date = last_date - pd.Timedelta(days=30)
baseline_end = start_analysis_date - pd.Timedelta(days=1)
baseline_start = baseline_end - pd.Timedelta(days=30)

df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

# ۵. اجرای تست ۳ برابر انحراف معیار (3-Sigma) برای سنسورهای تارگت
for col in target_sensors:
    mean_val = df_baseline[col].mean()
    std_val = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6

    upper_limit = mean_val + 3*std_val
    lower_limit = mean_val - 3*std_val

    # ایجاد ستون وضعیت برای هر سنسور تارگت در بازه تست
    df_fault[f'Status_{col}'] = df_fault[col].apply(
        lambda x: 'Fault' if (x > upper_limit or x < lower_limit) else 'Normal'
    )

# ۶. تابع محاسبه ماتریس همبستگی جزئی (Partial Correlation)
def get_partial_corr(data, columns):
    corr_matrix = data[columns].corr().values
    # افزودن مقدار کوچک برای جلوگیری از تکینگی (Regularization)
    precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
    d = np.sqrt(np.diag(precision))
    partial_corr = -precision / np.outer(d, d)
    np.fill_diagonal(partial_corr, 1.0)

    return pd.DataFrame(partial_corr, index=columns, columns=columns)

# ۷. محاسبات RCA (Root Cause Analysis)
pcorr_baseline = get_partial_corr(df_baseline, all_features)
pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
delta_pcorr = pcorr_fault - pcorr_baseline

# امتیاز انحراف (Deviation) و امتیاز تغییر شبکه (Network Change)
deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}

# نرمال‌سازی بین ۰ و ۱
def normalize(d):
    v = np.array(list(d.values()))
    if v.max() == v.min(): return {k: 0.5 for k in d.keys()}
    return {k: (val - v.min()) / (v.max() - v.min()) for k, val in d.items()}

dev_norm = normalize(deviation_scores)
chg_norm = normalize(change_scores)

# محاسبه امتیاز نهایی RCA (وزن ۵۰-۵۰ برای انحراف و تغییر ارتباطات)
rca_summary = pd.DataFrame({
    'Sensor': all_features,
    'RCA_Score': [ (0.5*dev_norm[c] + 0.5*chg_norm[c]) for c in all_features]
}).sort_values('RCA_Score', ascending=False).reset_index(drop=True)

rca_summary['Suspicion'] = rca_summary['RCA_Score'].apply(lambda x: 'HIGH' if x > 0.7 else 'MEDIUM' if x > 0.4 else 'LOW')

# ۸. ذخیره خروجی نهایی در اکسل (تک فایل با چند شیت)
try:
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # شیت ۱: تست ۳-سیگما برای یک ماه آخر (شامل ستون Date و سنسورهای تارگت)
        cols_to_save = target_sensors + [f'Status_{c}' for c in target_sensors]
        df_fault[cols_to_save].reset_index().to_excel(writer, sheet_name='Fault_Detection_Test', index=False)

        # شیت ۲: رتبه‌بندی ریشه عیب (RCA)
        rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)

        # شیت ۳: تغییرات همبستگی جزئی (Delta)
        delta_pcorr.to_excel(writer, sheet_name='Partial_Corr_Delta')

    print(f"\n🚀 تحلیل با موفقیت انجام شد و در مسیر زیر ذخیره شد:\n{output_filename}")
    print("\n--- اولویت‌بندی ریشه‌های احتمالی خطا (Top 5) ---")
    print(rca_summary.head(5).to_string(index=False))
    print(f"\n🎯 سنسور با بیشترین احتمال ریشه خطا: {rca_summary.iloc[0]['Sensor']}")

except Exception as e:
    print(f"❌ خطا در ذخیره‌سازی فایل اکسل: {e}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os
import warnings

# غیرفعال کردن هشدارهای غیرضروری
warnings.filterwarnings('ignore')

# ۱. تنظیمات مسیرها و متغیرها
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output1.xlsx'

os.makedirs(os.path.dirname(output_filename), exist_ok=True)

# لیست تمام سنسورها برای تحلیل شبکه ارتباطی
all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
                'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

# سنسورهایی که تست ۳-سیگما روی آن‌ها اجرا می‌شود
target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# ۲. خواندن داده‌ها
try:
    df_raw = pd.read_excel(file_path)
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.sort_values(by='date')
    print("✅ دیتا با موفقیت بارگذاری شد.")
except Exception as e:
    print(f"❌ خطا در خواندن فایل: {e}")
    exit()

# ۳. پیش‌پردازش و حذف داده‌های پرت (DBSCAN - حذف ۱۰ درصد داده‌های دور)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[all_features])

# خوشه‌بندی برای شناسایی نقاط پرت
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(scaled_data)

cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

def calculate_distance(i):
    label, point = labels[i], scaled_data[i].reshape(1, -1)
    if label != -1:
        return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
    return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

# حذف ۱۰ درصد داده‌های با بیشترین فاصله از خوشه‌ها
df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# ۴. جداسازی بازه نرمال (Baseline) و بازه تحلیل (Fault)
last_date = df_cleaned.index.max()
start_analysis_date = last_date - pd.Timedelta(days=30)
baseline_end = start_analysis_date - pd.Timedelta(days=1)
baseline_start = baseline_end - pd.Timedelta(days=30)

df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

# ۵. تست چندسطحی انحراف معیار (3-Sigma Rule)
def get_sigma_status(val, mean, std):
    if std == 0: return 'Normal'
    deviation = abs(val - mean) / std
    if deviation > 3:
        return 'Action Required'
    elif deviation > 2:
        return 'Warning'
    elif deviation > 1:
        return 'Normal (Minor Change)'
    else:
        return 'Normal'

for col in target_sensors:
    m = df_baseline[col].mean()
    s = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6
    df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))

# ۶. محاسبات همبستگی جزئی (Partial Correlation)
def get_partial_corr(data, columns):

    corr_matrix = data[columns].corr().values
    # Regularization برای جلوگیری از خطای ماتریس تکین
    precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
    d = np.sqrt(np.diag(precision))
    partial_corr = -precision / np.outer(d, d)
    np.fill_diagonal(partial_corr, 1.0)
    return pd.DataFrame(partial_corr, index=columns, columns=columns)

pcorr_baseline = get_partial_corr(df_baseline, all_features)
pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
delta_pcorr = pcorr_fault - pcorr_baseline

# ۷. محاسبه امتیازات RCA و رتبه‌بندی ریشه خطا
deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}

def normalize_dict(d):
    vals = np.array(list(d.values()))
    if vals.max() == vals.min(): return {k: 0.5 for k in d.keys()}
    return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}

dev_norm = normalize_dict(deviation_scores)
chg_norm = normalize_dict(change_scores)

rca_list = []
for c in all_features:
    strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
    # فرمول تجمیعی RCA Score
    score = (dev_norm[c]*0.5) + (chg_norm[c]*0.5)

    rca_list.append({
        'Sensor': c,
        'Change_Normalized': chg_norm[c],
        'Change_Score_Delta': change_scores[c],
        'Deviation_Normalized': dev_norm[c],
        'Deviation_Score': deviation_scores[c],
        'Strength_Score_Fault': strength_fault,
        'RCA_Score': score,
        'Root_Cause_Suspicion': score*(1 - strength_fault)
    })

df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)

# ۸. ذخیره‌سازی در اکسل با شیت‌های مجزا
try:
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # شیت ۱: جزئیات عیب‌یابی تک‌متغیره (Status)
        status_cols = [f'Status_{c}' for c in target_sensors]
        df_fault[target_sensors + status_cols].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)

        # شیت ۲: خلاصه تحلیل ریشه خطا
        df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)

        # شیت ۳: ماتریس تغییرات همبستگی
        delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')

    print(f"🚀 تحلیل با موفقیت پایان یافت. خروجی: {output_filename}")
    print("\n--- ۵ مظنون اصلی خرابی ---")
    print(df_rca_summary[['Sensor', 'RCA_Score', 'Root_Cause_Suspicion']].head(5))

except Exception as e:
    print(f"❌ خطا در ذخیره فایل: {e}")